# 03 - Hierarchical Flower Demo

This notebook runs the target three-layer topology:

1. Hospital SuperNodes train on local CIC-IDS2017 partitions.
2. Regional SuperLinks aggregate hospital updates into regional checkpoints.
3. RegionGateway SuperNodes expose only regional checkpoints to the global SuperLink.
4. The global SuperLink aggregates regional models into the global checkpoint.

In [ ]:
from pathlib import Path
import os

root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(root)
root

## 1. Generate and Validate Compose

The generated Compose file contains two regional SuperLinks, one global SuperLink, six hospital SuperNodes, and two gateway SuperNodes.

In [ ]:
# !python scripts/generate_compose.py
# !python scripts/configure_flower_profiles.py
# !docker compose config --quiet

## 2. Start the Simulated Infrastructure

For local development this uses Flower `--insecure`. Production-like deployments should add TLS, SuperNode authentication, and stronger network segmentation.

In [ ]:
# !docker compose up --build -d
# !docker compose ps

## 3. Run Hierarchical Training

The orchestrator launches one Flower run per region and one Flower run at the global layer for each global round. The global model is produced by the global Flower SuperLink, not by a plain Python averaging script.

In [ ]:
# !python scripts/run_hierarchical_rounds.py --global-rounds 3 --regional-rounds 2

## 4. Validate Artifacts

Each global round should produce regional checkpoints and a global checkpoint. `shared/` must not contain raw CSV/parquet data.

In [ ]:
checkpoints = sorted(Path('shared/checkpoints').rglob('*.pt'))
data_leaks = sorted(list(Path('shared').rglob('*.csv')) + list(Path('shared').rglob('*.parquet')))
checkpoints, data_leaks

## 5. Evaluate the Global Model

The evaluator reports per-hospital metrics and a global summary. Metrics include F1, ROC-AUC, AUPRC, false-positive rate, and false-negative rate.

In [ ]:
# !python scripts/evaluate_global_model.py --checkpoint shared/checkpoints/global/round_3.pt

In [ ]:
import pandas as pd
detail = Path('reports/metrics_summary.csv')
summary = Path('reports/metrics_summary_global.csv')
if summary.exists():
    display(pd.read_csv(summary))
elif detail.exists():
    display(pd.read_csv(detail).head())
else:
    print('Run hierarchical training and evaluation first.')